In [1]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-NhGB8OrVisfo8FihP8aoT3BlbkFJXYjQLEOchrvI9Hwg0XHl"

In [7]:
!pip install langchain_openai langchain_community duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun

In [12]:
search_tool = DuckDuckGoSearchRun()

In [13]:
llm = ChatOpenAI()

In [14]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

In [15]:
# Step 2 : Pull the ReAct prompt from Langchain hub
prompt = hub.pull('hwchase17/react')

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [16]:
@tool
def get_weather(city: str) -> str:

    """This is a function that gets the weather information of a given city"""
    api_key = '2f192ed601337bc46cf188b08f6e189e'  # Replace with your OpenWeatherMap API key
    base_url = 'http://api.openweathermap.org/data/2.5/weather'

    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'  # Use 'imperial' for Fahrenheit
    }

    response = requests.get(base_url, params=params)

    if response.status_code == 200:
        data = response.json()
        weather_details = {
            'city': data['name'],
            'temperature': data['main']['temp'],
            'weather': data['weather'][0]['description'],
            'humidity': data['main']['humidity'],
            'wind_speed': data['wind']['speed'],
        }
        return weather_details
    else:
        return {'error': 'City not found or API limit reached'}

# Example usage:
# print(get_weather('London'))

In [17]:
# Step 3: Create the React agent manually with the pulled prompt
agent = create_react_agent(
    llm = llm,
    tools = [search_tool, get_weather],
    prompt = prompt
)

In [18]:
# Calling the AgentExecutor
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool, get_weather],
    verbose=True
)

In [20]:
# Invoke the agent executor
response = agent_executor.invoke({"input":"Find the capital of Madhya Pradesh, and how is the weather in that city"})
print(response)



> Entering new AgentExecutor chain...
We can first search for the capital of Madhya Pradesh and then get the weather information for that city.
Action: duckduckgo_search
Action Input: "Capital of Madhya Pradesh"

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


No good DuckDuckGo Search Result was foundLet's try a different search query to get the capital of Madhya Pradesh.
Action: duckduckgo_search
Action Input: "Madhya Pradesh capital"Madhya Pradesh (/ ˌmɑːdjəprəˈdɛʃ /; 10 Hindi: [ˈmədʱjə pɾəˈdeːʃ] ⓘ; lit.'Central Province') is a state in central India. Its capital is Bhopal and the largest city is Indore. Other major cities includes … 2 days ago · Madhya Pradesh, state of India that is situated in the heart of the country. It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive … The official website of Madhya Pradesh Tourism that provides information on tourist places to visit, accommodations, culture, heritage, itineraries, etc. Madhya Pradesh is a state in the Republic of India. The state has an area of 119,016 sq mi (308,250 km 2). About 70,000,000 people are in Madhya Pradesh. In traditional Indian … Madhya Pradesh (MP) is situated in central India and is a part of the peninsular pl

/usr/local/lib/python3.11/dist-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Now that we know the capital of Madhya Pradesh is Bhopal, we can get the weather information for that city.
Action: get_weather
Action Input: Bhopal{'city': 'Bhopal', 'temperature': 31.13, 'weather': 'haze', 'humidity': 70, 'wind_speed': 3.6}I now know the final answer
Final Answer: The capital of Madhya Pradesh is Bhopal and the weather in that city is currently 31.13°C with haze, 70% humidity, and a wind speed of 3.6 km/h.

> Finished chain.
{'input': 'Find the capital of Madhya Pradesh, and how is the weather in that city', 'output': 'The capital of Madhya Pradesh is Bhopal and the weather in that city is currently 31.13°C with haze, 70% humidity, and a wind speed of 3.6 km/h.'}


In [21]:
response['output']

'The capital of Madhya Pradesh is Bhopal and the weather in that city is currently 31.13°C with haze, 70% humidity, and a wind speed of 3.6 km/h.'